In [ ]:
install.packages("janitor")
install.packages("flextable")
install.packages("officer")
# install.packages("openxlsx")
install.packages("paletteer")

In [ ]:
if (!require("BiocManager", quietly = TRUE))
    install.packages("BiocManager")

BiocManager::install("biomaRt")

In [ ]:
# Load the package
library(tidyverse)
library(ggplot2)
library(data.table)
library(purrr)
# Install these packages 
library(janitor)
library(flextable)
library(officer)
# library(openxlsx)
library(biomaRt)
library(patchwork)

In [ ]:
# Remove directories utlized in this script to ensure clean workspace 
# Recrate directories utlized in this script
system("rm -rf saige_gene_plot", intern=TRUE)
system("mkdir -p saige_gene_plot", intern=TRUE)

In [ ]:
# Connect to Ensembl
mart <- useMart("ensembl", dataset = "hsapiens_gene_ensembl")

In [ ]:

# Directory containing gene set-based rare variant assoication test results 
files <- list.files(path="merged_saige_gene_step02")
# Load file containing conversion from Ensembl gene IDs to HUGO gene symbols file 
# gene_symbols <- rename(fread("gene_symbols.tsv"), "Region" = "gene_id")

# Function: Formats the gene set-based rare variant assoication test results 
process_results <- function(results) { 
    results %>% 
    filter(Group != "Cauchy") %>% 
    # inner_join(gene_symbols, by = "Region") %>% 
    mutate(max_MAF = as.character(max_MAF)) %>% 
    mutate(negative_log_p_values = -log10(Pvalue)) %>% 
    mutate(Chromosome = gsub("chr", "", Chromosome)) %>% 
    mutate(Chromosome = as.numeric(Chromosome)) %>% 
    mutate(Region = as.factor(Region)) %>% 
    mutate(Region_factor = fct_reorder(Region, Chromosome))
  }

# Loop: Loads and format files 
for (f in files) {
  print(f)
  data <- fread(paste0("merged_saige_gene_step02/", f))
  processed <- process_results(data)
    
  # Loop through unique Ensembl IDs
  gene_info <- getBM(
    attributes = c("ensembl_gene_id", "hgnc_symbol"),
    filters = "ensembl_gene_id",
    values = unique(processed$Region),
    mart = mart
  )
  # Rename columns
  gene_info <- 
  rename(gene_info, 
       "Region" = "ensembl_gene_id", 
       "Symbol" = "hgnc_symbol" )
  # Join together Symbols with P-values 
  processed <- full_join(gene_info, processed, by ="Region")
  # Assign to the variable 
  assign(f, processed, envir = .GlobalEnv)
}

In [ ]:
# Set encode colors for the plots 
color_values <- c("01" = "#FCCC87FF",
                  "02" = "#8f5a16ff",
                  "03" = "#B37A32FF",
                  "11" = "#3FA3C5FF",
                  "12" = "#0F7BA2FF",
                  "13" = "#0B5A79FF")

In [ ]:
# Set encode colors for the plots 
color_values <- paletteer::paletteer_d("LaCroixColoR::Lemon")

In [ ]:
columns <- c("Region", "Symbol", "Group", "max_MAF", "negative_log_p_values")
first_results <- EUR_aml_phenotype_high_confidence_merged_chr_deprivation %>%
           dplyr::select(all_of(columns)) %>%
           dplyr::rename(Negative_Log_Deprivation = negative_log_p_values)

second_results <- EUR_aml_phenotype_high_confidence_merged_chr_deprivation_index %>%
           dplyr::select(all_of(columns)) %>%
           dplyr::rename(Negative_Log_Deprivation_Index = negative_log_p_values)

results_joined <- 
inner_join(first_results, second_results,
           by = c("Region", "Group", "max_MAF", "Symbol"))

In [ ]:
# Compute Spearman correlation
spearman <- cor(results_joined$Negative_Log_Deprivation,
                results_joined$Negative_Log_Deprivation_Index, 
                method = "spearman")

# Create a new column for color
results_joined$point_color <- ifelse(results_joined$Negative_Log_Deprivation_Index > 
                                     results_joined$Negative_Log_Deprivation, 
                                     "#F5D000FF", "#172869FF")

# Plot
library(ggplot2)

correlation_plot <- 
ggplot(results_joined, aes(x = Negative_Log_Deprivation, 
                           y = Negative_Log_Deprivation_Index)) + 
  geom_point(color = results_joined$point_color) + 
  geom_abline(intercept = 0, slope = 1, color = "#DD5129FF", linetype = "dashed", linewidth = 1.2) +
  labs(title = paste0("Spearman Correlation: ", round(spearman, 3)), 
       x = "Deprivation Index Components (-log10(p-values))", 
       y = "Deprivation Index (-log10(p-values))") +
  theme_bw()


options(repr.plot.height = 9, repr.plot.width = 24)

In [ ]:
# Save plot
ggsave(
    filename = paste0("saige_gene_plot/correlation_high_confidence_plot.png"), 
    plot = correlation_plot, height = 9, width = 9, dpi = 300
)